In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = sns.load_dataset('iris')

In [6]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [7]:
df['species'].unique()

array(['setosa', 'versicolor', 'virginica'], dtype=object)

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X = df.drop('species', axis=1)
y = df['species']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [11]:
from sklearn.neighbors import KNeighborsClassifier

In [12]:
model_knn = KNeighborsClassifier(n_neighbors=5)

In [13]:
model_knn.fit(X_train, y_train)

KNeighborsClassifier()

In [14]:
model_knn.score(X_test, y_test)

0.98

In [15]:
# also checking score with svm
from sklearn.svm import SVC

In [16]:
model_svm = SVC(C =10, kernel='linear', gamma='auto')

In [17]:
model_svm.fit(X_train, y_train)

SVC(C=10, gamma='auto', kernel='linear')

In [18]:
model_svm.score(X_test, y_test)

1.0

*Now using Grid Search CV*

In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
classifier = GridSearchCV((model_svm), {
    'C' : [1, 10, 20, 30],
    'kernel' : ['rbf', 'linear'],
}, cv=5, return_train_score=False)

In [21]:
classifier.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(C=10, gamma='auto', kernel='linear'),
             param_grid={'C': [1, 10, 20, 30], 'kernel': ['rbf', 'linear']})

In [22]:
classifier.cv_results_

{'mean_fit_time': array([0.00349422, 0.00272141, 0.00280657, 0.00373945, 0.00425839,
        0.00245285, 0.00350938, 0.00329952]),
 'std_fit_time': array([0.00073986, 0.00046955, 0.00059634, 0.00163363, 0.00179389,
        0.00011065, 0.00168218, 0.00174213]),
 'mean_score_time': array([0.00259929, 0.00218091, 0.00217166, 0.00307083, 0.00403361,
        0.00189981, 0.00301142, 0.00202522]),
 'std_score_time': array([2.44977075e-04, 2.54689685e-04, 4.13504980e-05, 1.04680055e-03,
        2.82914398e-03, 2.25642528e-04, 1.14046688e-03, 3.50448212e-04]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20, 30, 30],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear',
                    'rbf', 'linear'],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [

In [23]:
results = pd.DataFrame(classifier.cv_results_)

In [24]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003494,0.000740,0.002599,0.000245,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.95,0.90,0.9,1.00,0.95,0.94,0.037417,2
1,0.002721,0.000470,0.002181,0.000255,1,linear,"{'C': 1, 'kernel': 'linear'}",1.00,0.90,0.9,1.00,0.95,0.95,0.044721,1
2,0.002807,0.000596,0.002172,0.000041,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.95,0.80,0.9,1.00,1.00,0.93,0.074833,4
3,0.003739,0.001634,0.003071,0.001047,10,linear,"{'C': 10, 'kernel': 'linear'}",0.95,0.85,0.9,1.00,0.95,0.93,0.050990,6
4,0.004258,0.001794,0.004034,0.002829,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.95,0.80,0.9,1.00,0.95,0.92,0.067823,8
5,0.002453,0.000111,0.001900,0.000226,20,linear,"{'C': 20, 'kernel': 'linear'}",0.95,0.85,0.9,0.95,1.00,0.93,0.050990,6
6,0.003509,0.001682,0.003011,0.001140,30,rbf,"{'C': 30, 'kernel': 'rbf'}",0.95,0.80,0.9,1.00,1.00,0.93,0.074833,4
7,0.003300,0.001742,0.002025,0.000350,30,linear,"{'C': 30, 'kernel': 'linear'}",0.95,0.85,0.9,1.00,1.00,0.94,0.058310,3


In [26]:
results[['param_C', 'param_kernel', 'mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.94
1,1,linear,0.95
2,10,rbf,0.93
3,10,linear,0.93
4,20,rbf,0.92
5,20,linear,0.93
6,30,rbf,0.93
7,30,linear,0.94


**Now GRID CV ON KNN**

In [27]:
classifier_knn = GridSearchCV((model_knn), {
    'n_neighbors' : [1, 5, 10, 15],
    'weights' : ['uniform', 'distance'],
    'algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute']
}, cv = 5, return_train_score = False)

In [28]:
classifier_knn.fit(X, y)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': [1, 5, 10, 15],
                         'weights': ['uniform', 'distance']})

In [29]:
classifier_knn.cv_results_

{'mean_fit_time': array([0.00257311, 0.00233374, 0.0021102 , 0.00222135, 0.00214953,
        0.00220871, 0.00320268, 0.00234747, 0.00285029, 0.00224895,
        0.00406117, 0.00241551, 0.00237789, 0.00223722, 0.00261717,
        0.00213284, 0.00220466, 0.00236511, 0.00599036, 0.00257034,
        0.00235963, 0.00228715, 0.00293674, 0.002387  , 0.00229764,
        0.0024178 , 0.00290804, 0.00397091, 0.0021666 , 0.00344028,
        0.00251951, 0.00237975]),
 'std_fit_time': array([2.16662527e-04, 2.10024798e-04, 4.83159589e-05, 1.63274015e-04,
        1.70109102e-04, 1.68693571e-04, 1.82464864e-03, 1.15750391e-04,
        7.43927663e-04, 2.90903746e-04, 1.47669995e-03, 1.67411041e-04,
        1.92835128e-04, 9.02682590e-05, 3.65069449e-04, 2.13773517e-04,
        7.81432669e-05, 5.97252579e-05, 2.43239348e-03, 6.52896818e-05,
        2.58993522e-04, 2.19992377e-04, 7.77372639e-04, 4.89287903e-05,
        6.23461621e-05, 3.14237243e-04, 1.29075620e-03, 1.94285504e-03,
        5.31707857e-0

In [30]:
results_knn = pd.DataFrame(classifier_knn.cv_results_)

In [31]:
results_knn

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_algorithm,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002573,0.000217,0.003853,0.001104,auto,1,uniform,"{'algorithm': 'auto', 'n_neighbors': 1, 'weigh...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,25
1,0.002334,0.000210,0.003212,0.001007,auto,1,distance,"{'algorithm': 'auto', 'n_neighbors': 1, 'weigh...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,25
2,0.002110,0.000048,0.003336,0.000175,auto,5,uniform,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.966667,1.000000,0.933333,0.966667,1.0,0.973333,0.024944,9
3,0.002221,0.000163,0.002416,0.000092,auto,5,distance,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.966667,1.000000,0.900000,0.966667,1.0,0.966667,0.036515,17
4,0.002150,0.000170,0.003095,0.000137,auto,10,uniform,"{'algorithm': 'auto', 'n_neighbors': 10, 'weig...",0.966667,1.000000,1.000000,0.933333,1.0,0.980000,0.026667,6
5,0.002209,0.000169,0.002650,0.000173,auto,10,distance,"{'algorithm': 'auto', 'n_neighbors': 10, 'weig...",0.966667,1.000000,1.000000,0.966667,1.0,0.986667,0.016330,1
6,0.003203,0.001825,0.004474,0.001925,auto,15,uniform,"{'algorithm': 'auto', 'n_neighbors': 15, 'weig...",0.933333,1.000000,0.933333,0.966667,1.0,0.966667,0.029814,17
7,0.002347,0.000116,0.002621,0.000213,auto,15,distance,"{'algorithm': 'auto', 'n_neighbors': 15, 'weig...",0.966667,1.000000,0.933333,0.966667,1.0,0.973333,0.024944,9
8,0.002850,0.000744,0.004001,0.000988,ball_tree,1,uniform,"{'algorithm': 'ball_tree', 'n_neighbors': 1, '...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,25
9,0.002249,0.000291,0.002754,0.000688,ball_tree,1,distance,"{'algorithm': 'ball_tree', 'n_neighbors': 1, '...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,25


In [33]:
results_knn[['param_n_neighbors', 'param_weights', 'mean_test_score']].head()

,param_n_neighbors,param_weights,mean_test_score
0,1,uniform,0.960000
1,1,distance,0.960000
2,5,uniform,0.973333
3,5,distance,0.966667
4,10,uniform,0.980000


**Applying Random Search CV**

In [34]:
from sklearn.model_selection import RandomizedSearchCV

In [37]:
classifier_r = RandomizedSearchCV((model_svm), {
    'C' : [1, 10, 20, 30],
    'kernel' : ['rbf', 'linear'],
}, n_iter = 4, cv=5, return_train_score=False)

In [39]:
classifier_r.fit(X,y)

RandomizedSearchCV(cv=5, estimator=SVC(C=10, gamma='auto', kernel='linear'),
                   n_iter=4,
                   param_distributions={'C': [1, 10, 20, 30],
                                        'kernel': ['rbf', 'linear']})

In [40]:
results_r = pd.DataFrame(classifier_r.cv_results_)

In [42]:
# here it wasnt needed but use where thousands of parameters
results_r[['param_C', 'param_kernel', 'mean_test_score']]
# best perform 0	rbf	10	0.980000

,param_C,param_kernel,mean_test_score
0,20,rbf,0.966667
1,10,linear,0.973333
2,20,linear,0.966667
3,10,rbf,0.980000
